# Develop

## Import

In [1]:
import numpy as np
import pandas as pd
import json
import os
import time
import tempfile
import shutil

import pyarrow as pa
import pyarrow.parquet as pq

from pyspark.sql import SparkSession
from pyspark.sql.window import Window
from pyspark.sql.functions import col, lag, row_number, lit

from catboost import CatBoostClassifier, Pool
from sklearn.metrics import average_precision_score

from typing import List

## GRID SEARCH FUCKING SHIT

In [2]:
def fucking_grid_search(
        cols_json_file: str = '..\datasets\joined\columns_list.json',
        path_to_train_data_shit: str = None):
    """
    ОПИСАНИЕ ЕМАЕ
    """
# 1. Загружаем данные (лучше использовать фильтры или выборку, если файл гигантский)
    with open(cols_json_file, 'r') as file:
        cols = json.load(file)
    
    cat_cols = cols['cat']
    feat_cols = cols['all']
    train_df = pd.read_parquet(path_to_train_data_shit)

    
    for col in cat_cols:
        # train_df[col] = train_df[col].astype(int)
        train_df[col] = train_df[col].fillna(-1)
        
    X = train_df.drop('target', axis=1)[feat_cols]
    y = train_df['target']
    train_pool = Pool(X, y, cat_features=cat_cols,)
    
    # 2. Инициализируем модель с поддержкой GPU
    model = CatBoostClassifier(
        task_type='GPU',
        eval_metric='F1',         # По чему выбираем лучшее дерево и останавливаемся
        auto_class_weights='SqrtBalanced',
        # class_weights={0:1,1:500}, # МАГИЯ: CatBoost сам взвесит классы
        devices='0',       # Номер вашей видеокарты
        iterations=1000,   # Ограничьте итерации для скорости поиска
        early_stopping_rounds=150,
        logging_level='Info',
        bootstrap_type='MVS', # минимум вараенс семпл (объекты с большим градиентом выбираются чаще, чем легкие объекты частого класса)
    )

    # 3. Описываем сетку параметров
    # Для больших данных не делайте сетку слишком широкой
    grid = {
        'depth': [10, 12, 15, 20],
        'l2_leaf_reg': [1, 100, 1000, 2000],
        # 'scale_pos_weight': [50,100,250,500,750,1000, 1300], # подставьте свои пропорции
        'learning_rate': [0.03, 0.1],
        'min_data_in_leaf':[1, 100]
    }
    # 4. Запускаем поиск
    # search_by_train_test_split=True критически важен для больших данных (быстрее, чем CV)
    grid_search_result = model.grid_search(
        grid,
        train_pool,
        # subsample=0.1,
        plot=True
    )

    print(f"Лучшие параметры: {grid_search_result['params']}")
    best_params = grid_search_result['params']
    return best_params

In [3]:
# best_params = fucking_grid_search(path_to_train_data_shit = r'..\datasets\joined\train_data.parquet')
best_params = {
        'depth': 15,
        'l2_leaf_reg': 1000,
        'learning_rate': 0.1,
    }

In [4]:
def train_class_splitting(files_list: List[str] = None,
                         cols_json_file: str = None,
                         output_dir: str = '../datasets/split/',
                         batchsize: int = 100000,
                         mode: str = 'validation'):
    """
    Функция для разделения данных на два файла по целевому классу.
    
    Parameters:
    -----------
    files_list : List[str]
        Список путей к parquet файлам для разделения
    cols_json_file : str
        Путь до json файла с информацией по колонкам
    output_dir : str
        Директория для сохранения разделенных файлов
    batchsize : int
        Размер чанка для обработки
    """
    if files_list is None:
        raise AttributeError("Надо указать files_list - файл(ы) для разделения")
    if cols_json_file is None:
        raise AttributeError("Надо указать cols_json_file - json файл с инфой по колонкам")
    
    # Создаем выходную директорию, если её нет
    os.makedirs(output_dir, exist_ok=True)
    
    # Загружаем информацию о колонках
    with open(cols_json_file, 'r') as file:
        cols = json.load(file)
    
    target_col = cols['target']
    id_col = cols['id']
    
    # Пути для выходных файлов
    if mode == 'validation':
        output_file_0 = os.path.join(output_dir, 'train_0.parquet')
        output_file_1 = os.path.join(output_dir, 'train_1.parquet')
    elif mode == 'submit':
        output_file_0 = os.path.join(output_dir, 'train_0_submit.parquet')
        output_file_1 = os.path.join(output_dir, 'train_1_submit.parquet')
    
    # Временные файлы для безопасной записи
    temp_file_0 = os.path.join(output_dir, 'train_0_temp.parquet')
    temp_file_1 = os.path.join(output_dir, 'train_1_temp.parquet')
    
    # Удаляем временные файлы, если они существуют
    for temp_file in [temp_file_0, temp_file_1]:
        if os.path.exists(temp_file):
            try:
                os.remove(temp_file)
            except PermissionError:
                print(f"Предупреждение: не удалось удалить временный файл {temp_file}")
                time.sleep(1)
                try:
                    os.remove(temp_file)
                except:
                    pass
    
    # Удаляем выходные файлы, если они существуют
    for out_file in [output_file_0, output_file_1]:
        if os.path.exists(out_file):
            try:
                os.remove(out_file)
            except PermissionError:
                print(f"Предупреждение: не удалось удалить файл {out_file}")
                time.sleep(1)
                try:
                    os.remove(out_file)
                except:
                    pass
    
    # Счетчики для статистики
    total_count_0 = 0
    total_count_1 = 0
    
    print(f"Начинаем разделение данных по классам...")
    print(f"Класс 0 будет сохранен в: {output_file_0}")
    print(f"Класс 1 будет сохранен в: {output_file_1}")
    print("=" * 60)
    
    # Списки для накопления данных перед записью
    buffer_0 = []
    buffer_1 = []
    buffer_size = batchsize // 10  # Буфер на 10% от batchsize
    
    for file_idx, file in enumerate(files_list):
        print(f"\nОбработка файла {file_idx + 1}/{len(files_list)}: {file}")
        
        current_file = pq.ParquetFile(file)
        total_rows = current_file.metadata.num_rows
        total_batches = (total_rows + batchsize - 1) // batchsize
        
        print(f"В файле {total_rows} строк. Обработка {total_batches} батчей")
        
        for batch_num, batch in enumerate(current_file.iter_batches(batch_size=batchsize)):
            print(f"  Батч {batch_num + 1}/{total_batches}: чтение...", end=" ")
            chunk = batch.to_pandas()
            
            # Разделяем данные по целевому классу
            chunk_0 = chunk[chunk[target_col] == 0]
            chunk_1 = chunk[chunk[target_col] == 1]
            
            count_0 = len(chunk_0)
            count_1 = len(chunk_1)
            
            total_count_0 += count_0
            total_count_1 += count_1
            
            print(f"класс 0: {count_0}, класс 1: {count_1}")
            
            # Добавляем в буферы
            if count_0 > 0:
                buffer_0.append(chunk_0[[id_col] + cols['all'] + [target_col]])
            if count_1 > 0:
                buffer_1.append(chunk_1[[id_col] + cols['all'] + [target_col]])
            
            # Записываем буферы, если они накопились
            current_buffer_size_0 = sum(len(df) for df in buffer_0) if buffer_0 else 0
            current_buffer_size_1 = sum(len(df) for df in buffer_1) if buffer_1 else 0
            
            if buffer_0 and (current_buffer_size_0 >= buffer_size or batch_num == total_batches - 1):
                print(f"    Запись {current_buffer_size_0} строк класса 0...", end=" ")
                df_buffer_0 = pd.concat(buffer_0, ignore_index=True)
                
                # Для первой записи используем временный файл
                if not os.path.exists(output_file_0):
                    # Первая запись - создаем новый файл через временный
                    df_buffer_0.to_parquet(temp_file_0, compression='snappy', index=False)
                    os.rename(temp_file_0, output_file_0)
                else:
                    # Последующие записи - читаем, добавляем, сохраняем через временный
                    existing = pd.read_parquet(output_file_0)
                    combined = pd.concat([existing, df_buffer_0], ignore_index=True)
                    combined.to_parquet(temp_file_0, compression='snappy', index=False)
                    
                    # Удаляем старый файл и переименовываем временный
                    if os.path.exists(output_file_0):
                        os.remove(output_file_0)
                    os.rename(temp_file_0, output_file_0)
                
                buffer_0 = []  # Очищаем буфер
                print(f"OK")
            
            if buffer_1 and (current_buffer_size_1 >= buffer_size or batch_num == total_batches - 1):
                print(f"    Запись {current_buffer_size_1} строк класса 1...", end=" ")
                df_buffer_1 = pd.concat(buffer_1, ignore_index=True)
                
                # Для первой записи используем временный файл
                if not os.path.exists(output_file_1):
                    # Первая запись - создаем новый файл через временный
                    df_buffer_1.to_parquet(temp_file_1, compression='snappy', index=False)
                    os.rename(temp_file_1, output_file_1)
                else:
                    # Последующие записи - читаем, добавляем, сохраняем через временный
                    existing = pd.read_parquet(output_file_1)
                    combined = pd.concat([existing, df_buffer_1], ignore_index=True)
                    combined.to_parquet(temp_file_1, compression='snappy', index=False)
                    
                    # Удаляем старый файл и переименовываем временный
                    if os.path.exists(output_file_1):
                        os.remove(output_file_1)
                    os.rename(temp_file_1, output_file_1)
                
                buffer_1 = []  # Очищаем буфер
                print(f"OK")
    
    # Даем время на освобождение файлов
    time.sleep(1)
    
    # Проверяем итоговое количество записей
    try:
        final_count_0 = len(pd.read_parquet(output_file_0)) if os.path.exists(output_file_0) else 0
    except Exception as e:
        print(f"Ошибка при чтении файла класса 0: {e}")
        final_count_0 = total_count_0
    
    try:
        final_count_1 = len(pd.read_parquet(output_file_1)) if os.path.exists(output_file_1) else 0
    except Exception as e:
        print(f"Ошибка при чтении файла класса 1: {e}")
        final_count_1 = total_count_1
    
    # Удаляем временные файлы, если они остались
    for temp_file in [temp_file_0, temp_file_1]:
        if os.path.exists(temp_file):
            try:
                os.remove(temp_file)
            except:
                pass
    
    print("\n" + "=" * 60)
    print("Разделение завершено!")
    print(f"Всего записей класса 0: {final_count_0}")
    print(f"Всего записей класса 1: {final_count_1}")
    print(f"Общее количество записей: {final_count_0 + final_count_1}")
    if final_count_1 > 0:
        print(f"Соотношение классов: 1 : {final_count_0/final_count_1:.2f}")
    
    return output_file_0, output_file_1

In [5]:
# Исправленная версия train_in_chunks с правильной работой с parquet
def train_in_chunks(cols_json_file: str = None,
                    batchsize_0: int = 45000,
                    model: CatBoostClassifier = None,
                    auto_nan_filler: bool = True,
                    class_0_file: str = None,
                    class_1_file: str = None):
    """
    Функция для сбалансированного обучения модели по чанкам.
    """
    if class_0_file is None or class_1_file is None:
        raise AttributeError("Надо указать class_0_file и class_1_file - файлы с разделенными по классам данными")
    if cols_json_file is None:
        raise AttributeError("Надо указать cols_json_file - json файл с инфой по колонкам")
    if model is None:
        raise AttributeError("Надо указать model - действующую модель")
    
    with open(cols_json_file, 'r') as file:
        cols = json.load(file)
    
    feature_cols = cols['all']
    cat_cols = cols['cat']
    target_col = cols['target']
    
    # Открываем оба файла
    file_0 = pq.ParquetFile(class_0_file)
    file_1 = pq.ParquetFile(class_1_file)
    
    total_0 = file_0.metadata.num_rows
    total_1 = file_1.metadata.num_rows
    
    # Рассчитываем количество батчей и размер батча для класса 1
    batch_count = (total_0 + batchsize_0 - 1) // batchsize_0
    batchsize_1 = (total_1 + batch_count - 1) // batch_count
    
    print(f"Файл класса 0: {total_0} строк")
    print(f"Файл класса 1: {total_1} строк")
    print(f"Количество батчей: {batch_count}")
    print(f"Размер батча для класса 0: {batchsize_0}")
    print(f"Размер батча для класса 1: {batchsize_1}")
    print(f"Общий размер батча: ~{batchsize_0 + batchsize_1} строк")
    print(f"Соотношение классов в батче: 1 : {batchsize_0/batchsize_1:.2f}")
    print("=" * 60)
    
    # Создаем итераторы для обоих файлов
    iter_0 = file_0.iter_batches(batch_size=batchsize_0)
    iter_1 = file_1.iter_batches(batch_size=batchsize_1)
    
    for batch_num in range(batch_count):
        print(f"\nСтарт обработки {batch_num + 1}/{batch_count} батча")
        
        # Получаем батч из класса 0
        try:
            chunk_0 = next(iter_0).to_pandas()
        except StopIteration:
            print(f"Закончились данные класса 0 на батче {batch_num + 1}")
            break
        
        # Получаем батч из класса 1
        try:
            chunk_1 = next(iter_1).to_pandas()
        except StopIteration:
            print(f"Закончились данные класса 1 на батче {batch_num + 1}")
            chunk_1 = pd.DataFrame()
        
        # Объединяем данные
        chunk = pd.concat([chunk_0, chunk_1], ignore_index=True)
        
        # Перемешиваем
        chunk = chunk.sample(frac=1).reset_index(drop=True)
        
        print(f"В батче: класс 0 - {len(chunk_0)} строк, класс 1 - {len(chunk_1)} строк")
        print(f"Всего в батче: {len(chunk)} строк")
        
        if auto_nan_filler:
            for col in cat_cols:
                if col in chunk.columns:
                    chunk[col] = chunk[col].fillna('NaN').astype(str)
        
        X_batch = chunk[feature_cols]
        y_batch = chunk[target_col]
        train_pool = Pool(X_batch, y_batch, cat_features=cat_cols)
        
        print(f"Старт обучения {batch_num + 1}/{batch_count} батча")
        if batch_num == 0:
            model.fit(train_pool)
        else:
            model.fit(train_pool, init_model=model)
        print(f"Обучение {batch_num + 1}/{batch_count} батча завершено")
        print("==========================================================")
    
    # Безопасное сохранение модел
    # Создаем директорию для моделей, если её нет
    models_dir = os.path.abspath('../models')
    os.makedirs(models_dir, exist_ok=True)
    
    model_path = os.path.join(models_dir, 'catboost_model_balanced.cbm')
    
    # Пробуем разные способы сохранения
    print("\nСохраняем модель...")
    
    # Способ 1: Сохраняем через временный файл с простым именем
    try:
        # Используем временный файл с простым именем в текущей директории
        temp_dir = tempfile.mkdtemp()
        temp_model_path = os.path.join(temp_dir, 'model.cbm')
        
        # Сохраняем во временный файл
        model.save_model(temp_model_path)
        
        # Копируем в целевую директорию
        shutil.copy2(temp_model_path, model_path)
        
        # Удаляем временную директорию
        shutil.rmtree(temp_dir)
        
        print(f"Модель успешно сохранена в: {model_path}")
        
    except Exception as e1:
        print(f"Первый способ сохранения не сработал: {e1}")
        
        # Способ 2: Сохраняем в формате JSON
        try:
            json_path = os.path.join(models_dir, 'catboost_model_balanced.json')
            model.save_model(json_path, format='json')
            print(f"Модель сохранена в формате JSON: {json_path}")
            
        except Exception as e2:
            print(f"Второй способ сохранения не сработал: {e2}")
            
            # Способ 3: Сохраняем через pickle
            try:
                import pickle
                pkl_path = os.path.join(models_dir, 'catboost_model_balanced.pkl')
                with open(pkl_path, 'wb') as f:
                    pickle.dump(model, f)
                print(f"Модель сохранена через pickle: {pkl_path}")
                
            except Exception as e3:
                print(f"Третий способ сохранения не сработал: {e3}")
                print("Не удалось сохранить модель ни одним способом")
    
    print(f"\nОбучение полностью завершено. Всего обработано батчей: {batch_num + 1}")
    
    return model

In [6]:
valid_train_data_path = '../datasets/joined/train_data.parquet'

# Сначала разделяем данные
class_0_file, class_1_file = train_class_splitting(
    files_list=[valid_train_data_path],
    cols_json_file='../datasets/joined/columns_list.json',
    output_dir='../datasets/split/'
)

# Затем обучаем на сбалансированных батчах
catboost_model = CatBoostClassifier(
    **best_params,
    auto_class_weights='SqrtBalanced',
    early_stopping_rounds=100,   # Остановка, когда начнется переобучение
    verbose = 0,
    eval_metric='Logloss',

)
# # Затем обучаем на сбалансированных батчах
# catboost_model = CatBoostClassifier(
#     iterations=2000,
#     learning_rate=0.01,         # Маленький шаг — плавнее вероятности
#     depth=8,                    # Неглубокие деревья
#     l2_leaf_reg=400,             # Сильная регуляризация
#     eval_metric='Logloss',      # Логлосс сам по себе штрафует за уверенные ошибки
#     early_stopping_rounds=100,   # Остановка, когда начнется переобучение
#     auto_class_weights='Balanced',
#     verbose = 0
# )

train_in_chunks(
    cols_json_file='../datasets/joined/columns_list.json',
    batchsize_0=90000,  # Размер батча для класса 0
    model=catboost_model,
    class_0_file=class_0_file,
    class_1_file=class_1_file
)

Начинаем разделение данных по классам...
Класс 0 будет сохранен в: ../datasets/split/train_0.parquet
Класс 1 будет сохранен в: ../datasets/split/train_1.parquet

Обработка файла 1/1: ../datasets/joined/train_data.parquet
В файле 7176877 строк. Обработка 72 батчей
  Батч 1/72: чтение... класс 0: 99930, класс 1: 70
    Запись 99930 строк класса 0... OK
  Батч 2/72: чтение... класс 0: 99926, класс 1: 74
    Запись 99926 строк класса 0... OK
  Батч 3/72: чтение... класс 0: 99911, класс 1: 89
    Запись 99911 строк класса 0... OK
  Батч 4/72: чтение... класс 0: 99907, класс 1: 93
    Запись 99907 строк класса 0... OK
  Батч 5/72: чтение... класс 0: 99889, класс 1: 111
    Запись 99889 строк класса 0... OK
  Батч 6/72: чтение... класс 0: 99916, класс 1: 84
    Запись 99916 строк класса 0... OK
  Батч 7/72: чтение... класс 0: 99914, класс 1: 86
    Запись 99914 строк класса 0... OK
  Батч 8/72: чтение... класс 0: 99907, класс 1: 93
    Запись 99907 строк класса 0... OK
  Батч 9/72: чтение... 

KeyboardInterrupt: 

## Prediction

In [ ]:
def save_csv_results(ids, preds, save_path, chunk_num):
    """Вспомогательная функция для дозаписи в CSV"""
    # Создаем директорию, если её нет
    import os
    save_dir = os.path.dirname(save_path)
    if save_dir and not os.path.exists(save_dir):
        os.makedirs(save_dir, exist_ok=True)
        print(f"Создана директория: {save_dir}")
    
    res_df = pd.DataFrame({
        'event_id': ids,
        'predict': preds
    })
    
    # Если это первый чанк, создаем файл (w), если нет - дописываем (a)
    mode = 'w' if chunk_num == 0 else 'a'
    header = True if chunk_num == 0 else False
    
    res_df.to_csv(save_path, mode=mode, header=header, index=False)

def save_file_results(id_events: pd.Series,
                      predictions: np.ndarray,
                      save_path: str,
                      chunk_num: int):
    """
    Сохранение предсказаний вместе с идентификаторами в Parquet файл
    """
    # Создаем директорию, если её нет
    import os
    save_dir = os.path.dirname(save_path)
    if save_dir and not os.path.exists(save_dir):
        os.makedirs(save_dir, exist_ok=True)
        print(f"Создана директория: {save_dir}")
    
    result_df = pd.DataFrame({
        'event_id': id_events,
        'predict': predictions
    })
    
    # Конвертируем в Arrow Table
    table = pa.Table.from_pandas(result_df)
    
    if chunk_num == 0:
        # Первый чанк - создаем новый файл
        pq.write_table(
            table,
            save_path,
            compression='snappy',
            flavor='spark'
        )
        print(f"Создан файл: {save_path} ({len(result_df)} записей)")
    else:
        # Добавляем данные в существующий файл
        # Для parquet используем другой подход
        if os.path.exists(save_path):
            # Читаем существующий файл
            existing_df = pd.read_parquet(save_path)
            # Объединяем с новыми данными
            combined_df = pd.concat([existing_df, result_df], ignore_index=True)
            # Сохраняем обратно
            combined_df.to_parquet(save_path, compression='snappy', index=False)
        else:
            # Если файла нет, просто сохраняем
            result_df.to_parquet(save_path, compression='snappy', index=False)
        
        print(f"Добавлено {len(result_df)} записей в {save_path}")

def predict_in_chunks(file: str = None,
                      batchsize: int = 20000,
                      cols_json_file: str = None,
                      return_type: str = 'proba',
                      model = None,
                      save_path: str = None,
                      auto_nan_filler: bool = True,
                      use_parquet: bool = False  # Флаг для выбора формата сохранения
                      ):
    """
    Функция для предсказания по чанкам с автоматическим созданием директорий
    """
    if any(v is None for v in [file, cols_json_file, model, save_path]):
        raise AttributeError("Проверь входные аргументы: file, cols_json_file, model, save_path")
    
    # Создаем директорию для сохранения, если её нет
    import os
    save_dir = os.path.dirname(save_path)
    if save_dir and not os.path.exists(save_dir):
        os.makedirs(save_dir, exist_ok=True)
        print(f"Создана директория для сохранения: {save_dir}")
    
    with open(cols_json_file, 'r', encoding='utf-8') as f:
        cols = json.load(f)
    
    feature_cols = cols['all']
    id_col = cols['id']
    
    print(f"Обработка файла: {file}")
    
    current_file = pq.ParquetFile(file)
    total_rows = current_file.metadata.num_rows
    total_batches = (total_rows + batchsize - 1) // batchsize
    
    for batch_idx, batch in enumerate(current_file.iter_batches(batch_size=batchsize)):
        print(f'Старт {batch_idx + 1}/{total_batches}')
        
        df_chunk = batch.to_pandas()
        # Выбираем фичи и ID
        chunk = df_chunk[feature_cols].copy() 
        id_events = df_chunk[id_col]
        
        if auto_nan_filler:
            # Заполняем NaN и приводим к int для числовых колонок
            for col in chunk.columns:
                if chunk[col].dtype in ['float64', 'float32']:
                    chunk[col] = chunk[col].fillna(-1).astype(int)
                else:
                    chunk[col] = chunk[col].fillna('NaN').astype(str)

        if return_type == 'proba':
            preds = model.predict_proba(chunk)[:, 1]
        else:
            preds = model.predict(chunk)
        
        # Сохраняем текущий чанк в зависимости от выбранного формата
        if use_parquet:
            save_file_results(id_events, preds, save_path, chunk_num=batch_idx)
        else:
            save_csv_results(id_events, preds, save_path, chunk_num=batch_idx)
        
        print(f'Батч {batch_idx + 1} готов')

    print(f"Готово! Результаты тут: {save_path}")

In [ ]:
valid_predict_data_path = '../datasets/joined/valid_data.parquet'
valid_predict_saving_path = '../datasets/validation/validation_result.csv'  # или .parquet

predict_in_chunks(
    file=valid_predict_data_path,
    cols_json_file='../datasets/joined/columns_list.json',
    model=catboost_model,
    save_path=valid_predict_saving_path,
    use_parquet=False  # True для parquet, False для CSV
)

Обработка файла: ../datasets/joined/valid_data.parquet
Старт 1/90
Батч 1 готов
Старт 2/90
Батч 2 готов
Старт 3/90
Батч 3 готов
Старт 4/90
Батч 4 готов
Старт 5/90
Батч 5 готов
Старт 6/90
Батч 6 готов
Старт 7/90
Батч 7 готов
Старт 8/90
Батч 8 готов
Старт 9/90
Батч 9 готов
Старт 10/90
Батч 10 готов
Старт 11/90
Батч 11 готов
Старт 12/90
Батч 12 готов
Старт 13/90
Батч 13 готов
Старт 14/90
Батч 14 готов
Старт 15/90
Батч 15 готов
Старт 16/90
Батч 16 готов
Старт 17/90
Батч 17 готов
Старт 18/90
Батч 18 готов
Старт 19/90
Батч 19 готов
Старт 20/90
Батч 20 готов
Старт 21/90
Батч 21 готов
Старт 22/90
Батч 22 готов
Старт 23/90
Батч 23 готов
Старт 24/90
Батч 24 готов
Старт 25/90
Батч 25 готов
Старт 26/90
Батч 26 готов
Старт 27/90
Батч 27 готов
Старт 28/90
Батч 28 готов
Старт 29/90
Батч 29 готов
Старт 30/90
Батч 30 готов
Старт 31/90
Батч 31 готов
Старт 32/90
Батч 32 готов
Старт 33/90
Батч 33 готов
Старт 34/90
Батч 34 готов
Старт 35/90
Батч 35 готов
Старт 36/90
Батч 36 готов
Старт 37/90
Батч 37 готов
С

## Error Analysis

In [ ]:
# Кастомная реализация sklearn.average_precision_score для big data
def average_precision_score(predict_path: str,
                            answers_path: str,
                            predict_column: str = 'target',
                            answers_column: str = 'target',
                            id_column: str = 'event_id'
                            ) -> float:
    """
    Расчет Average Precision (AP) метрики на больших данных с использованием Spark.
    
    AP = Σ (R_n - R_{n-1}) * P_n, где:
    - P_n - precision на n-ом пороге
    - R_n - recall на n-ом пороге
    
    predict_path: путь до parquet файла с предсказаниями (вероятности от 0 до 1)
    answers_path: путь до parquet файла с правильными ответами (бинарные метки 0/1)
    predict_column: название колонки с предсказаниями
    answers_column: название колонки с правильными ответами
    id_column: название колонки с идентификатором события
    
    Возвращает float: значение Average Precision метрики
    """
    
    spark = SparkSession.builder \
        .appName("AveragePrecisionCalculatorOptimized") \
        .config("spark.sql.adaptive.enabled", "true") \
        .getOrCreate()

    try:
        # Загружаем и объединяем данные
        predictions = spark.read.parquet(predict_path)
        answers = spark.read.parquet(answers_path)
        
        predict_cols = predictions.columns
        answers_cols = answers.columns
        if id_column not in predict_cols:
            raise ValueError(f"В файле предсказаний отсутствует колонка: {id_column}")
        if id_column not in answers_cols:
            raise ValueError(f"В файле ответов отсутствует колонка: {id_column}")
        if predict_column not in predict_cols:
            raise ValueError(f"В файле предсказаний отсутствует колонка: {predict_column}")
        if answers_column not in answers_cols:
            raise ValueError(f"В файле ответов отсутствует колонка: {answers_column}")
        
        
        merged = (predictions
                    .select(id_column, predict_column)
                    .join(answers.select(id_column, answers_column),
                        on=id_column,
                        how="inner")
                    .cache())
        
        total_count = merged.count()
        total_positives = merged.filter(col(answers_column) == 1).count()
        
        if total_positives == 0:
            return 0.0
        
        # Определяем окно для сортировки по убыванию предсказаний
        window_spec = Window.orderBy(col(predict_column).desc())
        
        # Добавляем индикаторы и кумулятивные суммы
        ap_data = merged.withColumn(
            "is_positive", col(answers_column)
        ).withColumn(
            "cumulative_tp", sum("is_positive").over(window_spec)
        ).withColumn(
            "row_num", row_number().over(window_spec)
        )
        
        # Расчет precision и recall для каждой строки
        # Используем lag для получения предыдущих значений
        result = ap_data.withColumn(
            "precision", col("cumulative_tp") / col("row_num")
        ).withColumn(
            "prev_tp", lag("cumulative_tp", 1, 0).over(window_spec)
        ).withColumn(
            "recall_diff", 
            (col("cumulative_tp") - col("prev_tp")) / lit(total_positives)
        ).agg(
            sum(col("recall_diff") * col("precision")).alias("average_precision")
        ).collect()[0][0]
        
        merged.unpersist()
        
        return float(result if result is not None else 0.0)
        
    finally:
        spark.stop()

In [ ]:
def target_class_mistakes_finder(predict_path: str,
                                 answers_path: str,
                                 predict_column: str = 'predict',
                                 answers_column: str = 'target',
                                 id_column: str = 'event_id', 
                                 target_class: int = 1):
    """
    Универсальная функция для поиска ошибок, работающая с CSV и Parquet
    """
    import os
    
    # Создаем Spark сессию
    spark = SparkSession.builder \
        .appName("FindFalseNegatives") \
        .config("spark.sql.adaptive.enabled", "true") \
        .getOrCreate()

    try:
        # Функция для загрузки данных в зависимости от расширения
        def load_data(path):
            if path.endswith('.csv'):
                return spark.read.option("header", "true").csv(path)
            elif path.endswith('.parquet'):
                return spark.read.parquet(path)
            else:
                # Пробуем определить формат по содержимому
                try:
                    return spark.read.parquet(path)
                except:
                    return spark.read.option("header", "true").csv(path)
        
        print(f"Загрузка предсказаний из: {predict_path}")
        predictions = load_data(predict_path)
        
        print(f"Загрузка ответов из: {answers_path}")
        answers = load_data(answers_path)
        
        # Приводим типы данных
        from pyspark.sql.types import DoubleType, IntegerType
        from pyspark.sql.functions import col
        
        # Для CSV все колонки могут быть строками, конвертируем
        if predict_path.endswith('.csv'):
            predictions = predictions.withColumn(predict_column, 
                                                col(predict_column).cast(DoubleType()))
        
        if answers_path.endswith('.csv'):
            answers = answers.withColumn(answers_column, 
                                        col(answers_column).cast(IntegerType()))
        
        # Для предсказаний конвертируем в бинарные метки (0 или 1)
        # Используем порог 0.5 для конвертации вероятностей в классы
        predictions = predictions.withColumn(
            predict_column + "_binary",
            (col(predict_column) > 0.5).cast(IntegerType())
        )
        
        predict_cols = predictions.columns
        answers_cols = answers.columns
        
        # Проверяем наличие колонок
        if id_column not in predict_cols:
            raise ValueError(f"В файле предсказаний отсутствует колонка: {id_column}")
        if id_column not in answers_cols:
            raise ValueError(f"В файле ответов отсутствует колонка: {id_column}")
        if answers_column not in answers_cols:
            raise ValueError(f"В файле ответов отсутствует колонка: {answers_column}")
        
        # Объединяем данные
        merged = (predictions
                  .select(id_column, predict_column + "_binary")
                  .withColumnRenamed(predict_column + "_binary", "pred_class")
                  .join(answers.select(id_column, answers_column),
                        on=id_column,
                        how="inner")
                  )
        
        # Определяем условие для поиска ошибок
        if target_class in [0, 1]:
            error_condition = (col(answers_column) == target_class) & (col("pred_class") != target_class)
        else:
            raise ValueError("target_class должен быть 0, 1")
        
        errors = merged.filter(error_condition)
        
        # Считаем статистику
        total_count = merged.count()
        mismatch_count = merged.filter(col("pred_class") != col(answers_column)).count()
        error_count = errors.count()
        
        print(f"\nСтатистика:")
        print(f"  Всего записей: {total_count}")
        print(f"  Всего несоответствий: {mismatch_count} ({mismatch_count/total_count*100:.2f}%)")
        print(f"  Найдено целевых ошибок (False Negatives): {error_count}")
        
        if error_count > 0:
            print(f"\nПервые 10 {id_column} с ошибками (FN):")
            errors.select(id_column).show(10)

        return [row[id_column] for row in errors.select(id_column).collect()]
        
    except Exception as e:
        print(f"Ошибка при выполнении: {e}")
        raise
    finally:
        spark.stop()

In [ ]:
# Поиск лучших параметров и отбор полезных признаков (shap?)

In [ ]:
with open('../datasets/backward_errors.csv', 'w') as f:
    back_ward_errors = target_class_mistakes_finder(predict_path=valid_predict_saving_path,
                             answers_path=valid_predict_data_path)
    f.write('event_id\n')
    for error in back_ward_errors:
        f.write(error+'\n')

Загрузка предсказаний из: ../datasets/validation/validation_result.csv
Загрузка ответов из: ../datasets/joined/valid_data.parquet

Статистика:
  Всего записей: 1795327
  Всего несоответствий: 1603 (0.09%)
  Найдено целевых ошибок (False Negatives): 1229

Первые 10 event_id с ошибками (FN):
+---------------+
|       event_id|
+---------------+
|123131715673876|
|123157484130117|
|123174663485280|
|123183255586662|
|123200433277647|
|123209026628461|
|123226202872383|
|123234792783871|
|123243382277029|
|123251974873165|
+---------------+
only showing top 10 rows


## Получение важности признаков

In [ ]:
model_for_weights= CatBoostClassifier()
model_for_weights.load_model('../models/catboost_model_balanced.cbm')
feature_importance = model_for_weights.get_feature_importance()
feature_names = model_for_weights.feature_names_

# Создаем DataFrame для удобства
df_importance = pd.DataFrame({
    'feature': feature_names,
    'importance': feature_importance
}).sort_values(by='importance', ascending=False)

print(df_importance) # Топ-10 признаков
print(list(df_importance['feature']))

                       feature  importance
25              mean_operation   18.757434
19               user_activity   15.105015
10                        hour    8.460069
4          count_ops_last_week    6.429613
8         median_ops_last_week    6.239673
28                    is_night    5.621821
0             operaton_amt_log    5.375811
29        inter_quantile_range    5.026937
13                  ops_sum_2h    4.558988
27                  max_amt_2h    4.515035
23                 day_of_week    4.379374
11            median_operation    4.339856
1                  compromised    3.788132
20           avg_ops_last_week    2.947250
15                ops_count_2h    2.067091
5              currency_iso_cd    0.365763
2   channel_indicator_sub_type    0.289409
24                      pos_cd    0.283994
30           language_category    0.265092
31      channel_indicator_type    0.259907
9                event_type_nm    0.240622
7           web_rdp_connection    0.189805
22         

## Test Predict (подготовка решения)

In [ ]:
submit_save_path = '../submit/submit_16_03.csv'

In [ ]:
# train_files = [valid_train_data_path, valid_predict_data_path]

# # Сначала разделяем данные
# class_0_file, class_1_file = train_class_splitting(
#     files_list=train_files,
#     cols_json_file='../datasets/joined/columns_list.json',
#     output_dir='../datasets/split/',
#     mode='submit'
# )

# # Затем обучаем на сбалансированных батчах
# catboost_model = CatBoostClassifier(
#     iterations=2000,
#     learning_rate=0.01,         # Маленький шаг — плавнее вероятности
#     depth=8,                    # Неглубокие деревья
#     l2_leaf_reg=400,             # Сильная регуляризация
#     eval_metric='Logloss',      # Логлосс сам по себе штрафует за уверенные ошибки
#     early_stopping_rounds=100,   # Остановка, когда начнется переобучение
#     auto_class_weights='Balanced',
#     verbose = 0
# )

# train_in_chunks(
#     cols_json_file='../datasets/joined/columns_list.json',
#     batchsize_0=90000,  # Размер батча для класса 0
#     model=catboost_model,
#     class_0_file=class_0_file,
#     class_1_file=class_1_file
# )

predict_in_chunks(file='../datasets/joined/test_data.parquet',
                  cols_json_file='../datasets/joined/columns_list.json',
                  model=catboost_model,
                  save_path=submit_save_path)

Обработка файла: ../datasets/joined/test_data.parquet
Старт 1/32
Батч 1 готов
Старт 2/32
Батч 2 готов
Старт 3/32
Батч 3 готов
Старт 4/32
Батч 4 готов
Старт 5/32
Батч 5 готов
Старт 6/32
Батч 6 готов
Старт 7/32
Батч 7 готов
Старт 8/32
Батч 8 готов
Старт 9/32
Батч 9 готов
Старт 10/32
Батч 10 готов
Старт 11/32
Батч 11 готов
Старт 12/32
Батч 12 готов
Старт 13/32
Батч 13 готов
Старт 14/32
Батч 14 готов
Старт 15/32
Батч 15 готов
Старт 16/32
Батч 16 готов
Старт 17/32
Батч 17 готов
Старт 18/32
Батч 18 готов
Старт 19/32
Батч 19 готов
Старт 20/32
Батч 20 готов
Старт 21/32
Батч 21 готов
Старт 22/32
Батч 22 готов
Старт 23/32
Батч 23 готов
Старт 24/32
Батч 24 готов
Старт 25/32
Батч 25 готов
Старт 26/32
Батч 26 готов
Старт 27/32
Батч 27 готов
Старт 28/32
Батч 28 готов
Старт 29/32
Батч 29 готов
Старт 30/32
Батч 30 готов
Старт 31/32
Батч 31 готов
Старт 32/32
Батч 32 готов
Готово! Результаты тут: ../submit/submit_15_01.csv


In [ ]:

def submit_checker(path: str = None):
    result_df = pd.read_csv(path)
    
    row_count = len(result_df)
    if row_count != 633683:
        raise ValueError(f"Представленный датафрейм имеет иное количество строк: {row_count}. Такой сабмит не пройдет. Необходимо 633683 строк")
    
    if list(result_df.columns) != ['event_id', 'predict']:
        raise ValueError(f'Неверные названия столбиков: {result_df.columns}')

    print('All good. СЛАВА БЛИНАМ И БЛИННОМУ ТИМЛИДУ!')
    
submit_checker(submit_save_path)

All good. СЛАВА БЛИНАМ И БЛИННОМУ ТИМЛИДУ!
